In [448]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [449]:
def f(x):
    return 3*x**2 - 4*x + 5

In [450]:
f(3.0)

20.0

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward(): # backwards requires += because of the multivariate case and the possibility of multi-use
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __radd__(self, other): # other + self
        return self + other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out
    
    
    def __rmul__(self, other): # other * self
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other): # self / other
        return self * other**-1

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
            
        out._backward = _backward
        return out
    
    def exp(self):
        v = np.exp(self.data)
        out = Value(v, (self, ), 'exp')

        def _backward():
            self.grad += v * out.grad
    
        out._backward = _backward
        return out

    def backward(self):
        topo : list[Value] = []
        visited = set()

        def build_topo(curr_node : Value):
            if curr_node not in visited:
                visited.add(curr_node)
                for child in curr_node._prev:
                    build_topo(child)
                topo.append(curr_node)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [452]:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735, label='b')
# x1 * w1 + x2 * w2 + b

x1w1 = x1 * w1; x1w1.label = "x1*w1"
x2w2 = x2 * w2; x2w2.label = "x2*w2"
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = "x1*w1 + x2*w2"
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh()
o

Value(data=0.7071067376767731)

In [453]:
w1 / w2

Value(data=-3.0)

In [454]:
o.grad = 1.0

In [455]:
o.backward()

In [456]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]
    
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

x = [2.0, 3.0, -1]
n = MLP(3, [4, 4, 1])
n(x)

Value(data=0.383896751741999)

In [457]:
len(n.parameters())

41

In [458]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [459]:
for k in range(20):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    # backward pass
    for p in n.parameters():
        p.grad = 0

    loss.backward()

    for p in n.parameters():
        p.data += -0.1 * p.grad
    
    print(k, loss.data)
    

0 5.1019729606947815
1 2.6984352354873082
2 1.462190550623931
3 0.9133805156553529
4 0.7197920110137506
5 0.17086428359891898
6 0.08989086424736559
7 0.07256273756919827
8 0.061456850498717346
9 0.053414196507596545
10 0.04724588994130859
11 0.04234278562372752
12 0.038344418139264105
13 0.03501905416378984
14 0.0322093032264826
15 0.029803897173254837
16 0.02772168978586158
17 0.025901994455949838
18 0.024298464331976515
19 0.022875067001436784


In [460]:
ypred

[Value(data=0.928351730070433),
 Value(data=-0.9009736537272554),
 Value(data=-0.9440592757904998),
 Value(data=0.9306746039584064)]